# Architecture Overview: FK-Diffusion-Steering for Stable Diffusion

This codebase implements Feynman-Kac Diffusion Steering (FK Steering) for Stable Diffusion models. It's essentially an extension of Hugging Face's Diffusers library that adds particle-based guidance during the diffusion process.

## Core Components

### 1. Standard Diffusion Pipeline Class

The codebase provides a main pipeline class that extend the original Stable Diffusion pipelines:

- **`FKDStableDiffusion`** (fkd_pipeline_sd.py): For Stable Diffusion v1.x and v2.x models

Inherit from their respective base classes in Diffusers and extend them with FK Steering capabilities.

### 2. Stable Diffusion Architecture

The standard Stable Diffusion components are preserved:

- **VAE** (`AutoencoderKL`): Encodes images to latent space and decodes latents back to images
- **Text Encoder** (`CLIPTextModel`): Processes text prompts into embeddings
- **UNet** (`UNet2DConditionModel`): The core denoising model
- **Scheduler** (typically `DDIMScheduler`): Controls the denoising trajectory
- **Additional components for SDXL**: Second text encoder, added conditioning for resolution

### 3. FK Steering Implementation

The FK Steering mechanism is implemented in `fkd_class.py` with the `FKD` class:





This is the core innovation, using a particle-based approach to guide the diffusion process toward higher-reward outputs.

## Pipeline Flow

The modified generation pipeline follows this flow:

1. **Initialization**:
   - Load the base Stable Diffusion models
   - Encode the text prompt
   - Initialize random latents

2. **FK Steering Setup**:
   - Create multiple particles (copies of the latents)
   - Define reward functions for guiding generation
   - Set up resampling parameters (`lmbda`, `resample_frequency`, etc.)

3. **Denoising Loop with FK Steering**:
   - For each timestep:
     - Apply the UNet to predict noise
     - Apply classifier-free guidance
     - Update latents using the scheduler
     - **FK Steering**: At specified timesteps, the system:
       - Decodes current latents to images
       - Evaluates images with reward functions
       - Resamples particles based on reward scores

4. **Final Output**:
   - Decode final latents to images
   - Return highest-scoring images

## Key Modifications to Standard Diffusion

The main changes to standard diffusion are in the denoising loop:



In [ ]:
# Standard diffusion just does:
latents = self.scheduler.step(noise_pred, t, latents, **extra_step_kwargs, return_dict=False)[0]

# FK Steering version does:
step_dict = self.scheduler.step(noise_pred, t, latents, **extra_step_kwargs, return_dict=True)
latents = step_dict["prev_sample"]
x0_preds = step_dict["pred_original_sample"]

if fkd_args is not None and fkd_args["use_smc"]:
    latents, current_pop_images = fkd.resample(
        sampling_idx=i, latents=latents, x0_preds=x0_preds
    )



## Reward Functions

The system supports various reward functions:

- **ImageReward**: Evaluates image quality and prompt alignment
- **CLIP-Score**: Measures text-image alignment
- **Human Preference Models**: Uses preference models for guidance
- **LLM Grading**: Uses LLMs to evaluate images

## Usage Example

From playground_fksteering.ipynb, typical usage involves:



In [ ]:
# Configure FK Steering parameters
fkd_args = dict(
    lmbda=2.0,                  # Controls how strongly rewards influence resampling
    num_particles=4,            # Number of particles (samples) to maintain
    adaptive_resampling=True,   # Adapts resampling based on effective sample size
    resample_frequency=20,      # How often to resample
    time_steps=10,              # Total diffusion steps
    potential_type='max',       # Type of reward aggregation
    guidance_reward_fn='ImageReward',  # Which reward function to use
    use_smc=True,               # Enable Sequential Monte Carlo
)

# Run generation with FK Steering
images = pipeline(prompt, num_inference_steps=fkd_args["time_steps"], fkd_args=fkd_args)



This architecture allows FK Steering to enhance standard Stable Diffusion models to produce outputs that better satisfy user-defined reward criteria without any fine-tuning.